# Notebook 02 — CDC Event Analysis with DuckDB

This notebook analyses the `weather_forecasts_cdc` table that the
`kafka_cdc_to_duckdb` Airflow DAG populates from the Debezium CDC stream.

**What you will learn**
- How to download and open a DuckDB file from MinIO
- Running SQL queries on persistent DuckDB tables
- Analysing change-data-capture patterns (creates, updates, deletes)
- Joining CDC data with open-weather reference data inside DuckDB
- Scatter plots and violin plots with seaborn

**Pre-requisites**
- The `kafka_cdc_to_duckdb` DAG must have run at least once so that
  `weather-analytics/duckdb/weather.duckdb` exists in MinIO.
- `matplotlib` and `seaborn` must be installed in the Jupyter image.

---

In [ ]:
# Cell 1 — Imports
import sys
import os

SHARED_PATH = '/home/jovyan/work/shared'
if SHARED_PATH not in sys.path:
    sys.path.insert(0, SHARED_PATH)

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import duckdb

from minio_helper import get_client, object_exists

%matplotlib inline
sns.set_theme(style='whitegrid', palette='tab10')
plt.rcParams['figure.dpi'] = 120

# Path where we will download the DuckDB file from MinIO
DUCKDB_LOCAL = '/tmp/weather_notebook.duckdb'

print('Imports OK')

In [ ]:
# Cell 2 — Download DuckDB from MinIO
# We download a copy to /tmp so we can open it with duckdb.connect().
# The notebook only reads — it does NOT write back to MinIO.

BUCKET = 'weather-analytics'
OBJECT = 'duckdb/weather.duckdb'

client = get_client()

if not object_exists(client, BUCKET, OBJECT):
    raise FileNotFoundError(
        f'{BUCKET}/{OBJECT} not found. '
        'Run the kafka_cdc_to_duckdb Airflow DAG first.'
    )

client.fget_object(BUCKET, OBJECT, DUCKDB_LOCAL)
print(f'Downloaded DuckDB to {DUCKDB_LOCAL}')

# Open in read-only mode so we cannot accidentally corrupt the file
con = duckdb.connect(DUCKDB_LOCAL, read_only=True)
print('DuckDB connection open')

In [ ]:
# Cell 3 — Inspect available tables and row counts

tables = con.execute("SHOW TABLES").df()
print('Tables in DuckDB:')
display(tables)

# Row count for the CDC table
cdc_count = con.execute("SELECT COUNT(*) AS rows FROM weather_forecasts_cdc").fetchone()[0]
print(f'\nweather_forecasts_cdc: {cdc_count:,} rows')

In [ ]:
# Cell 4 — Preview the CDC table schema and first rows

print('=== Schema ===')
schema = con.execute("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_name = 'weather_forecasts_cdc'
    ORDER BY ordinal_position
""").df()
display(schema)

print('\n=== First 10 rows ===')
con.execute("""
    SELECT * FROM weather_forecasts_cdc
    ORDER BY event_ts DESC
    LIMIT 10
""").df()

In [ ]:
# Cell 5 — Descriptive statistics on temperature
# This gives us the basic shape of the forecast data in our app database.

stats = con.execute("""
    SELECT
        COUNT(*)                                    AS total_forecasts,
        COUNT(DISTINCT date)                        AS unique_dates,
        MIN(temperature_c)                          AS min_temp,
        MAX(temperature_c)                          AS max_temp,
        ROUND(AVG(temperature_c), 1)               AS avg_temp,
        ROUND(STDDEV(temperature_c), 1)            AS std_temp,
        COUNT(CASE WHEN summary IS NULL THEN 1 END) AS null_summary_count,
        COUNT(DISTINCT summary)                     AS unique_summaries
    FROM weather_forecasts_cdc
""").df()

display(stats)

In [ ]:
# Cell 6 — CDC operation breakdown
# How many creates, updates, and deletes are in our current dataset?

ops = con.execute("""
    SELECT
        op,
        CASE op
            WHEN 'c' THEN 'Create'
            WHEN 'u' THEN 'Update'
            WHEN 'd' THEN 'Delete'
            WHEN 'r' THEN 'Snapshot read'
            ELSE op
        END AS operation_label,
        COUNT(*) AS count
    FROM weather_forecasts_cdc
    GROUP BY op
    ORDER BY count DESC
""").df()

display(ops)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(ops['operation_label'], ops['count'])
ax.set_title('CDC Events by Operation Type', fontsize=12)
ax.set_xlabel('Operation')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 7 — Temperature distribution: histogram by summary label
# Does the 'Summary' field (Freezing, Bracing, Chilly, etc.) align with
# the recorded temperature? This is a data quality check.

cdc_df = con.execute("""
    SELECT temperature_c, summary, date
    FROM weather_forecasts_cdc
    WHERE summary IS NOT NULL
""").df()

if len(cdc_df) > 0:
    # Sort by median temperature so the legend is ordered cold → hot
    order = (
        cdc_df.groupby('summary')['temperature_c']
        .median()
        .sort_values()
        .index.tolist()
    )

    fig, ax = plt.subplots(figsize=(10, 5))
    sns.violinplot(
        data=cdc_df,
        x='summary',
        y='temperature_c',
        order=order,
        ax=ax,
        # hue maps colour to category — same as x here, just for colour
        hue='summary',
        hue_order=order,
        palette='coolwarm',
        legend=False,
    )
    ax.set_title('Temperature Distribution by Forecast Summary', fontsize=13)
    ax.set_xlabel('Summary')
    ax.set_ylabel('Temperature (°C)')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print('No CDC data with non-null summaries yet.')

In [ ]:
# Cell 8 — Daily summary view
# The DAG created a VIEW called daily_summary that aggregates CDC rows.
# Views in DuckDB are recomputed on each query — no stale data.

daily = con.execute("""
    SELECT * FROM daily_summary
    ORDER BY date
""").df()

if len(daily) > 0:
    daily['date'] = pd.to_datetime(daily['date'])

    fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

    axes[0].plot(daily['date'], daily['avg_temp_c'], marker='o', markersize=3)
    axes[0].fill_between(daily['date'], daily['min_temp_c'], daily['max_temp_c'], alpha=0.2)
    axes[0].set_title('App Forecast Temperature by Date (from CDC stream)', fontsize=12)
    axes[0].set_ylabel('Temperature (°C)')

    axes[1].bar(daily['date'], daily['forecast_count'], width=1)
    axes[1].set_title('Number of Forecasts per Date', fontsize=12)
    axes[1].set_ylabel('Forecast count')
    axes[1].set_xlabel('Date')

    plt.tight_layout()
    plt.show()
else:
    print('daily_summary view is empty — no CDC data loaded yet.')

In [ ]:
# Cell 9 — Scatter plot: app forecast temperature vs Open-Meteo reference
# This cross-source comparison checks how our app's forecasts relate to
# real-world weather. We load the Open-Meteo New York CSV from MinIO and
# join it with the CDC data on date.
#
# DATA: The app generates random WeatherForecasts, so correlation will be
# low — this cell demonstrates the technique, not a meaningful finding.
# Once minions use realistic weather profiles (Notebook 04), the
# correlation should improve significantly.

from minio_helper import read_csv as minio_read_csv

WEATHER_BUCKET = 'weather-raw'
METEO_OBJECT = 'open-meteo/new_york.csv'

if object_exists(client, WEATHER_BUCKET, METEO_OBJECT):
    # Load Open-Meteo reference data
    meteo_df = minio_read_csv(
        client, WEATHER_BUCKET, METEO_OBJECT,
        parse_dates=['date'],
    )

    # Load CDC data into a pandas DataFrame for the join
    cdc_for_join = con.execute("""
        SELECT id, date, temperature_c, summary
        FROM weather_forecasts_cdc
    """).df()
    cdc_for_join['date'] = pd.to_datetime(cdc_for_join['date'])

    # Inner join on date — only rows where both sources have data
    comparison = cdc_for_join.merge(
        meteo_df[['date', 'temperature_2m_mean']],
        on='date',
        how='inner',
    )

    if len(comparison) > 0:
        fig, ax = plt.subplots(figsize=(7, 6))
        ax.scatter(
            comparison['temperature_2m_mean'],
            comparison['temperature_c'],
            alpha=0.5,
            edgecolors='none',
        )
        # Perfect-agreement diagonal
        lim = [
            min(comparison['temperature_2m_mean'].min(), comparison['temperature_c'].min()) - 2,
            max(comparison['temperature_2m_mean'].max(), comparison['temperature_c'].max()) + 2,
        ]
        ax.plot(lim, lim, 'r--', linewidth=1, label='Perfect agreement')
        ax.set_xlim(lim)
        ax.set_ylim(lim)
        ax.set_title('App Forecast vs Actual Temperature (New York)', fontsize=12)
        ax.set_xlabel('Actual daily mean temperature (C) — Open-Meteo')
        ax.set_ylabel('App forecast temperature (C) — CDC stream')
        ax.legend()
        plt.tight_layout()
        plt.show()

        # Show correlation coefficient
        corr = comparison['temperature_c'].corr(comparison['temperature_2m_mean'])
        print(f'Pearson correlation: {corr:.3f}')
        print(f'(Low correlation is expected — minions generate random temperatures)')
    else:
        print('No matching dates between CDC forecasts and Open-Meteo data.')
        print('This is expected if the app uses forecast dates outside 2020-2024.')
else:
    print(f'{WEATHER_BUCKET}/{METEO_OBJECT} not found in MinIO.')
    print('Run the download_weather_datasets Airflow DAG first, then re-run this cell.')

In [ ]:
# Cell 10 — Close the connection and clean up
con.close()

import os
if os.path.exists(DUCKDB_LOCAL):
    os.remove(DUCKDB_LOCAL)
    print(f'Removed local copy: {DUCKDB_LOCAL}')

print('Done')